In [4]:
import os
import polars as pl
from pyprojroot import here

In [5]:
# load all file name from folder
def load_file_name(folder_path):
    file_name_list = []
    for file in os.listdir(folder_path):
        if file.endswith("txt") and "_" in file:
            file_name_list.append(file)
    return file_name_list


ls = load_file_name(here() / "data/raw")
ls

['RUL_FD001.txt',
 'RUL_FD002.txt',
 'RUL_FD003.txt',
 'RUL_FD004.txt',
 'test_FD001.txt',
 'test_FD002.txt',
 'test_FD003.txt',
 'test_FD004.txt',
 'train_FD001.txt',
 'train_FD002.txt',
 'train_FD003.txt',
 'train_FD004.txt']

In [6]:
def load_one_df(
    file_name,
    col_ls=(
        ["UNIT_NUM"]
        + ["CYCLE"]
        + ["OS" + "_" + str(i) for i in range(1, 4)]
        + ["SM" + "_" + str(i) for i in range(1, 22)]
    ),
):
    df = pl.read_csv(
        str(here()) + "/data/raw/" + file_name, separator=" ", has_header=False
    )
    df = df.select(
        pl.all().exclude([col for col in df.columns if df[col].null_count() == len(df)])
    )

    if len(df.columns) >= len(col_ls):
        df.columns = col_ls
        df = df.with_columns(pl.lit(file_name).alias("FILE_NAME"))
    else:
        df.columns = ["RUL"]
        df = df.with_row_index("UNIT_NUM", 1)

    return df

In [7]:
df_ls = [load_one_df(i) for i in ls]

In [8]:
RUL_1 = df_ls[0]
RUL_2 = df_ls[1]
RUL_3 = df_ls[2]
RUL_4 = df_ls[3]

RUL_1 = RUL_1.rename({"RUL": "MAX_RUL"})
RUL_2 = RUL_2.rename({"RUL": "MAX_RUL"})
RUL_3 = RUL_3.rename({"RUL": "MAX_RUL"})
RUL_4 = RUL_4.rename({"RUL": "MAX_RUL"})

test_1 = df_ls[4]
test_2 = df_ls[5]
test_3 = df_ls[6]
test_4 = df_ls[7]

train_1 = df_ls[8]
train_2 = df_ls[9]
train_3 = df_ls[10]
train_4 = df_ls[11]

# RUL

In [9]:
train_1 = train_1.with_columns(pl.col("CYCLE").max().over("UNIT_NUM").alias("MAX_RUL"))
train_2 = train_2.with_columns(pl.col("CYCLE").max().over("UNIT_NUM").alias("MAX_RUL"))
train_3 = train_3.with_columns(pl.col("CYCLE").max().over("UNIT_NUM").alias("MAX_RUL"))
train_4 = train_4.with_columns(pl.col("CYCLE").max().over("UNIT_NUM").alias("MAX_RUL"))

test_1 = test_1.join(RUL_1, on="UNIT_NUM", how="left")
test_2 = test_2.join(RUL_2, on="UNIT_NUM", how="left")
test_3 = test_3.join(RUL_3, on="UNIT_NUM", how="left")
test_4 = test_4.join(RUL_4, on="UNIT_NUM", how="left")

In [10]:
train_1 = train_1.with_columns((pl.col("MAX_RUL") - pl.col("CYCLE")).alias("RUL"))
train_2 = train_2.with_columns((pl.col("MAX_RUL") - pl.col("CYCLE")).alias("RUL"))
train_3 = train_3.with_columns((pl.col("MAX_RUL") - pl.col("CYCLE")).alias("RUL"))
train_4 = train_4.with_columns((pl.col("MAX_RUL") - pl.col("CYCLE")).alias("RUL"))

test_1 = test_1.with_columns((pl.col("MAX_RUL") - pl.col("CYCLE")).alias("RUL"))
test_2 = test_2.with_columns((pl.col("MAX_RUL") - pl.col("CYCLE")).alias("RUL"))
test_3 = test_3.with_columns((pl.col("MAX_RUL") - pl.col("CYCLE")).alias("RUL"))
test_4 = test_4.with_columns((pl.col("MAX_RUL") - pl.col("CYCLE")).alias("RUL"))

# Concat

In [11]:
train = pl.concat([train_1, train_2, train_3, train_4])
test = pl.concat([test_1, test_2, test_3, test_4])

In [13]:
train

UNIT_NUM,CYCLE,OS_1,OS_2,OS_3,SM_1,SM_2,SM_3,SM_4,SM_5,SM_6,SM_7,SM_8,SM_9,SM_10,SM_11,SM_12,SM_13,SM_14,SM_15,SM_16,SM_17,SM_18,SM_19,SM_20,SM_21,FILE_NAME,MAX_RUL,RUL
i64,i64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,i64,i64,f64,f64,f64,str,i64,i64
1,1,-0.0007,-0.0004,100.0,518.67,641.82,1589.7,1400.6,14.62,21.61,554.36,2388.06,9046.19,1.3,47.47,521.66,2388.02,8138.62,8.4195,0.03,392,2388,100.0,39.06,23.419,"""train_FD001.txt""",192,191
1,2,0.0019,-0.0003,100.0,518.67,642.15,1591.82,1403.14,14.62,21.61,553.75,2388.04,9044.07,1.3,47.49,522.28,2388.07,8131.49,8.4318,0.03,392,2388,100.0,39.0,23.4236,"""train_FD001.txt""",192,190
1,3,-0.0043,0.0003,100.0,518.67,642.35,1587.99,1404.2,14.62,21.61,554.26,2388.08,9052.94,1.3,47.27,522.42,2388.03,8133.23,8.4178,0.03,390,2388,100.0,38.95,23.3442,"""train_FD001.txt""",192,189
1,4,0.0007,0.0,100.0,518.67,642.35,1582.79,1401.87,14.62,21.61,554.45,2388.11,9049.48,1.3,47.13,522.86,2388.08,8133.83,8.3682,0.03,392,2388,100.0,38.88,23.3739,"""train_FD001.txt""",192,188
1,5,-0.0019,-0.0002,100.0,518.67,642.37,1582.85,1406.22,14.62,21.61,554.0,2388.06,9055.15,1.3,47.28,522.19,2388.04,8133.8,8.4294,0.03,393,2388,100.0,38.9,23.4044,"""train_FD001.txt""",192,187
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
249,251,9.9998,0.25,100.0,489.05,605.33,1516.36,1315.28,10.52,15.46,404.59,2319.66,8840.16,1.27,46.08,380.16,2388.73,8185.69,8.4541,0.03,372,2319,100.0,29.11,17.5234,"""train_FD004.txt""",255,4
249,252,0.0028,0.0015,100.0,518.67,643.42,1598.92,1426.77,14.62,21.57,567.59,2388.47,9117.12,1.31,48.04,535.02,2388.46,8185.47,8.2221,0.03,396,2388,100.0,39.38,23.7151,"""train_FD004.txt""",255,3
249,253,0.0029,0.0,100.0,518.67,643.68,1607.72,1430.56,14.62,21.57,569.04,2388.51,9126.53,1.31,48.24,535.41,2388.48,8193.94,8.2525,0.03,395,2388,100.0,39.78,23.827,"""train_FD004.txt""",255,2


In [14]:
test

UNIT_NUM,CYCLE,OS_1,OS_2,OS_3,SM_1,SM_2,SM_3,SM_4,SM_5,SM_6,SM_7,SM_8,SM_9,SM_10,SM_11,SM_12,SM_13,SM_14,SM_15,SM_16,SM_17,SM_18,SM_19,SM_20,SM_21,FILE_NAME,MAX_RUL,RUL
i64,i64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,i64,i64,f64,f64,f64,str,i64,i64
1,1,0.0023,0.0003,100.0,518.67,643.02,1585.29,1398.21,14.62,21.61,553.9,2388.04,9050.17,1.3,47.2,521.72,2388.03,8125.55,8.4052,0.03,392,2388,100.0,38.86,23.3735,"""test_FD001.txt""",112,111
1,2,-0.0027,-0.0003,100.0,518.67,641.71,1588.45,1395.42,14.62,21.61,554.85,2388.01,9054.42,1.3,47.5,522.16,2388.06,8139.62,8.3803,0.03,393,2388,100.0,39.02,23.3916,"""test_FD001.txt""",112,110
1,3,0.0003,0.0001,100.0,518.67,642.46,1586.94,1401.34,14.62,21.61,554.11,2388.05,9056.96,1.3,47.5,521.97,2388.03,8130.1,8.4441,0.03,393,2388,100.0,39.08,23.4166,"""test_FD001.txt""",112,109
1,4,0.0042,0.0,100.0,518.67,642.44,1584.12,1406.42,14.62,21.61,554.07,2388.03,9045.29,1.3,47.28,521.38,2388.05,8132.9,8.3917,0.03,391,2388,100.0,39.0,23.3737,"""test_FD001.txt""",112,108
1,5,0.0014,0.0,100.0,518.67,642.51,1587.19,1401.92,14.62,21.61,554.16,2388.01,9044.55,1.3,47.31,522.15,2388.03,8129.54,8.4031,0.03,390,2388,100.0,38.99,23.413,"""test_FD001.txt""",112,107
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
248,277,41.9991,0.8401,100.0,445.0,550.3,1364.4,1129.17,3.91,5.72,138.34,2212.35,8351.73,1.02,42.3,130.87,2388.5,8112.61,9.4427,0.02,331,2212,100.0,10.53,6.262,"""test_FD004.txt""",26,-251
248,278,20.0026,0.7005,100.0,491.19,608.0,1494.75,1260.88,9.35,13.66,334.75,2324.23,8758.69,1.07,44.53,314.51,2388.33,8086.83,9.2772,0.02,366,2324,100.0,24.33,14.6486,"""test_FD004.txt""",26,-252
248,279,34.9988,0.8413,100.0,449.44,555.92,1370.65,1130.97,5.48,8.0,194.92,2223.57,8370.49,1.02,42.33,182.76,2388.64,8100.84,9.3982,0.02,336,2223,100.0,14.69,8.8389,"""test_FD004.txt""",26,-253
